# **Importing and downloading the needed libraries**

In [ ]:
!pip install numpy==1.25.2

In [ ]:
!pip install gensim

In [ ]:
pip install fasttext

In [ ]:
import pandas as pd
import numpy as np
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, Dense
import fasttext
from keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import BatchNormalization
from keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
import tensorflow as tf
from transformers import create_optimizer
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer
from sklearn.metrics import classification_report

In [ ]:
!pip install tensorflow

In [ ]:
!pip install transformers

# **importing the preprocessed data**

In [ ]:
#the data is already preprocessed in the previous notebook so the only thing i did was downloading it and importing it here
df = pd.read_csv("/content/ready_data.csv")

In [ ]:
df.shape

In [ ]:
df.head(100)

In [ ]:
#splitting the text because the model only accepts it splited
df['x'] = df['x'].astype(str).apply(lambda x: x.split())

In [ ]:
#encoding the target
le = LabelEncoder()
y_encoded = le.fit_transform(df['y'])

# ***Neural network classification models***

# **preparing for RNN and LSTM**

In [ ]:
#downloading the pretrained word2vec(cbow) model
!wget https://archive.org/download/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip
!unzip full_uni_cbow_100_twitter.zip

In [ ]:
from gensim.models import Word2Vec
#loading the word2vec model using gensim and diving it the model we downloaded
word2vec_model = Word2Vec.load("full_uni_cbow_100_twitter.mdl")

In [ ]:
#downloading the pretrained arabic FastText model
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz

In [ ]:
#unzipping the pretrained arabic FastText model
!gunzip cc.ar.300.bin.gz

In [ ]:
#loading the FastText model "cc.ar.300.bin.gz"
Fasttext_model = fasttext.load_model('cc.ar.300.bin')

In [ ]:
#sequance embedding using fasttext where for each word a vector of (300) is returned
FastText_seq = []
for s in df['x']:
   FastText_seq.append(np.array([Fasttext_model.get_word_vector(word) for word in s]))

In [ ]:
#sequance embedding using word2vec where for each word a vector of (100) is returned
word2vec_seq = []
for s in df['x']:
  word2vec_seq.append(np.array([word2vec_model.wv[word] for word in s if word in word2vec_model.wv]))

In [ ]:
#padding the embeddings of word2vec and fasttext so they all have the same length where some sentances have less words that other
FastText_padded = pad_sequences(FastText_seq, padding='post', dtype='float32')
Word2Vec_padded = pad_sequences(word2vec_seq, padding='post', dtype='float32')

In [ ]:
#making sue that y and x have the same length
y = df['y'].values[:len(FastText_padded)]

In [ ]:
#splitting the prepared embeddings and encoded y
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(FastText_padded,y_encoded, test_size=0.2, random_state=42)
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(Word2Vec_padded, y_encoded, test_size=0.2, random_state=42)

In [ ]:
classes_num = 5

# **RNN with FastText**

In [ ]:
# initializing the model
model_RNN_FT = Sequential()
#adding a layer with SimpleRNN model
model_RNN_FT.add(SimpleRNN(64,input_shape=(X_train_ft.shape[1], X_train_ft.shape[2])))
#Add Dense layer with relu and softmax activation functions (softmax because we have multiclass classification)
model_RNN_FT.add(Dense(64, activation='relu'))
model_RNN_FT.add(Dense(classes_num, activation='softmax'))
# Compile and training the model using adam optimizer and sparse_categorical_crossentropy loss because our classification problem nature
model_RNN_FT.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_RNN_FT.summary()
#fitting the RNN model using fasttext training data
model_RNN_FT.fit(X_train_ft, y_train_ft, epochs=10, batch_size=16)
# Evaluating the model
loss, accuracy = model_RNN_FT.evaluate(X_test_ft, y_test_ft)
print(f"Test Accuracy: {accuracy:.4f}")


# **RNN with Word2vec**

In [ ]:
# initializing the model
model_RNN_W2V = Sequential()
#adding a layer with SimpleRNN model
model_RNN_W2V.add(SimpleRNN(64,input_shape=(X_train_w2v.shape[1], X_train_w2v.shape[2])))
#Add Dense layer with relu and softmax activation functions (softmax because we have multiclass classification)
model_RNN_W2V.add(Dense(64, activation='relu'))
model_RNN_W2V.add(Dense(classes_num, activation='softmax'))
# Compile and training the model using adam optimizer and sparse_categorical_crossentropy loss because our classification problem nature
model_RNN_W2V.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_RNN_W2V.summary()
#fitting the RNN model using fasttext training data
model_RNN_W2V.fit(X_train_w2v, y_train_w2v, epochs=10, batch_size=16)
# Evaluate
loss, accuracy = model_RNN_W2V.evaluate(X_test_w2v, y_test_w2v)
print(f"Test Accuracy: {accuracy:.4f}")


#**FastText emmbeding worked better than Word2vec that is why i will use it with LSTM**

In [ ]:
# initializing the model
model_LSTM = Sequential()
#adding a layer with LSTM model
model_LSTM.add(LSTM(64, input_shape=(X_train_ft.shape[1], X_train_ft.shape[2])))
#Add Dense layer with relu and softmax activation functions (softmax because we have multiclass classification)
model_LSTM.add(Dense(64, activation='relu'))
model_LSTM.add(Dense(classes_num, activation='softmax'))
# Compile and training the model using adam optimizer and sparse_categorical_crossentropy loss because our classification problem nature
model_LSTM.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_LSTM.summary()
#fitting the LSTM model using fasttext training data
model_LSTM.fit(X_train_ft, y_train_ft, epochs=10, batch_size=16)
# Evaluating the model
loss, accuracy = model_LSTM.evaluate(X_test_ft, y_test_ft)
print(f"Test Accuracy: {accuracy:.4f}")

# ***Transformers***

# **loading the original data because transformers deals with raw data**

In [ ]:
df_trans = pd.read_csv("/content/altibbi_specialty_data.csv")

In [ ]:
#the data is big so we took 25000 random samples of it
df_trans = df_trans.drop("name_ar", axis=1)
df_trans = df_trans.sample(n=25000, random_state=42)

# **BERT transformer**

**preparing the data for bert**

In [ ]:
#initializing arabbert Auto toknizer using("aubmindlab/bert-base-arabertv02")
bert_toknizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")
#encoding the target
df_trans['specialty_id'] = le.fit_transform(df_trans['specialty_id'])
#here we are transforming the data to a form that bert understands where we use padding to ensure that all the inputs have the same length
#ensuring that all texts are shorter than 130 tokens
#and lastly ensuring that the output works with tensorflow
encodings_tokenization_bert = bert_toknizer(list(df_trans["question_body"].astype(str)),padding=True,truncation=True,max_length=130,return_tensors='tf')
#attention mask that helps the model now the difference between padding tokens and actual tokens
attention_mask_bert = encodings_tokenization_bert['attention_mask'].numpy()
#input formed as toknized numpy array
input_bert = encodings_tokenization_bert['input_ids'].numpy()
#the target as numpy array
target = tf.convert_to_tensor(df_trans['specialty_id']).numpy()
#splitting and preparing testing and training data
#%80 training rest testing
X_train_in_bert, X_test_in_bert, X_train_attention_bert, X_test_attention_bert, y_train_bert, y_test_bert = train_test_split(input_bert, attention_mask_bert, target, test_size=0.2, random_state=42)
# Train and test datasets using tensor slices where batch = 8
train_bert = tf.data.Dataset.from_tensor_slices(({'input_ids': X_train_in_bert,'attention_mask': X_train_attention_bert}, y_train_bert)).batch(8)
test_bert = tf.data.Dataset.from_tensor_slices(({'input_ids': X_test_in_bert,'attention_mask': X_test_attention_bert}, y_test_bert)).batch(8)


In [ ]:
#Initial learning rate = init_lr=3e-5
#	num_train_steps = number of batches(8) × epochs(4)
optimizer_bert, schedule_bert = create_optimizer(init_lr=3e-5,num_warmup_steps=int(0.1*len(train_bert) * 4),num_train_steps=len(train_bert) * 4)

In [ ]:
#preparing the arabertmodel
#TFAutoModelForSequenceClassification is a tenserflow version of bert that is spicifcally for classification tasks
bert_model = TFAutoModelForSequenceClassification.from_pretrained("aubmindlab/bert-base-arabertv02", num_labels=5)

In [ ]:
#compiling the bert model using the optimizer we initialized and setting the loss function to SparseCategoricalCrossentropy which is user for classification task
bert_model.compile(optimizer=optimizer_bert,loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])

In [ ]:
#finaly fitting the model
bert_model.fit(train_bert, validation_data=test_bert, epochs=4)

In [ ]:
#bert predictions
bert_predictions = bert_model.predict(test_bert)
#logits are raw scores before applying softmax, where each with its corusponding class and sample
logits = bert_predictions.logits
#this returns the maximum logit which shows which class is the predicted
max_class = tf.argmax(logits, axis=1).numpy()
actual_y = []
for b in test_bert:
    _, labels = b
    np.array(actual_y.extend(labels.numpy()))

In [ ]:
print(classification_report(actual_y, max_class))